RE-REQ: **Run** _kqlqueryset-append.kql_ outside of the notebook first.

In [2]:
# Example of query for reading data from Kusto. Replace T with your <tablename>.
kustoQuery = "['MyTable'] | take 10"
# The query URI for reading the data e.g. https://<>.kusto.data.microsoft.com.
kustoUri = "https://trd-cff114afmpqwdjz7ux.z0.kusto.fabric.microsoft.com"
# The database with data to be read.
database = "EH1"
# The access credentials.
accessToken = mssparkutils.credentials.getToken(kustoUri)
kustoDf  = spark.read\
    .format("com.microsoft.kusto.spark.synapse.datasource")\
    .option("accessToken", accessToken)\
    .option("kustoCluster", kustoUri)\
    .option("kustoDatabase", database)\
    .option("kustoQuery", kustoQuery).load()

# Example that uses the result data frame.
kustoDf.show()

StatementMeta(, 89bb3201-2f97-4b1d-b6a2-ca9c076c03a8, 6, Finished, Available, Finished, False)

+---+---------+
| Id|    Value|
+---+---------+
|101|NewValue1|
|102|NewValue2|
|101|NewValue1|
|102|NewValue2|
+---+---------+



In [10]:
# Execute a Kusto management command (.set-or-append) via REST mgmt endpoint.
import json
import requests

kustoCommand = ".set-or-append MyTable <| datatable(Id:int, Value:string)[998, 'NewValue1']"

# Convert data endpoint to management endpoint host if needed.
mgmtUri = kustoUri.replace(".kusto.data.", ".kusto.")
mgmtEndpoint = f"{mgmtUri}/v1/rest/mgmt"

# Reuse token acquisition approach from Cell 1.
accessToken = mssparkutils.credentials.getToken(mgmtUri)

payload = {
    "db": database,
    "csl": kustoCommand
}

headers = {
    "Authorization": f"Bearer {accessToken}",
    "Content-Type": "application/json; charset=utf-8",
    "Accept": "application/json"
}

response = requests.post(mgmtEndpoint, headers=headers, data=json.dumps(payload), timeout=120)

print(f"Status: {response.status_code}")
if response.ok:
    print(".set-or-append executed successfully")
    print(response.text)
else:
    print("Command failed")
    print(response.text)
    response.raise_for_status()

StatementMeta(, 89bb3201-2f97-4b1d-b6a2-ca9c076c03a8, 103, Finished, Available, Finished, False)

Status: 200
.set-or-append executed successfully
{"Tables":[{"TableName":"Table_0","Columns":[{"ColumnName":"ExtentId","DataType":"Guid","ColumnType":"guid"},{"ColumnName":"OriginalSize","DataType":"Double","ColumnType":"real"},{"ColumnName":"ExtentSize","DataType":"Double","ColumnType":"real"},{"ColumnName":"CompressedSize","DataType":"Double","ColumnType":"real"},{"ColumnName":"IndexSize","DataType":"Double","ColumnType":"real"},{"ColumnName":"RowCount","DataType":"Int64","ColumnType":"long"}],"Rows":[["3d069f2d-edab-40f5-a1c6-c161c934f4ac",21.0,926.0,287.0,639.0,1]]},{"TableName":"Table_1","Columns":[{"ColumnName":"OperationId","DataType":"Guid","ColumnType":"guid"},{"ColumnName":"Database","DataType":"String","ColumnType":"string"},{"ColumnName":"Table","DataType":"String","ColumnType":"string"},{"ColumnName":"FailedOn","DataType":"DateTime","ColumnType":"datetime"},{"ColumnName":"IngestionSourcePath","DataType":"String","ColumnType":"string"},{"ColumnName":"Details","DataType":"St

In [12]:
# Example of query for reading data from Kusto. Replace T with your <tablename>.
kustoQuery = "['MyTable'] | take 10"
# The query URI for reading the data e.g. https://<>.kusto.data.microsoft.com.
kustoUri = "https://trd-cff114afmpqwdjz7ux.z0.kusto.fabric.microsoft.com"
# The database with data to be read.
database = "EH1"
# The access credentials.
accessToken = mssparkutils.credentials.getToken(kustoUri)
kustoDf  = spark.read\
    .format("com.microsoft.kusto.spark.synapse.datasource")\
    .option("accessToken", accessToken)\
    .option("kustoCluster", kustoUri)\
    .option("kustoDatabase", database)\
    .option("kustoQuery", kustoQuery).load()

# Example that uses the result data frame.
kustoDf.show()

StatementMeta(, 89bb3201-2f97-4b1d-b6a2-ca9c076c03a8, 106, Finished, Available, Finished, False)

+---+---------+
| Id|    Value|
+---+---------+
|998|NewValue1|
|101|NewValue1|
|102|NewValue2|
|101|NewValue1|
|102|NewValue2|
|998|NewValue1|
|998|NewValue1|
+---+---------+

